# ShallowSeek — Data-Scale and Runtime Ablations for Speculative Decoding

This notebook demonstrates the individual project contribution of Siyuan Zhang (`388143`). It is intended to run inside the RCP Jupyter environment launched by `notebooks/submit.sh`.

It integrates two parts of my work:

- **Training-data scale ablation with the HF/manual loop:** training and evaluating Qwen2.5 draft models on original UltraChat 10k vs 50k data across CE, FKL, RKL, and JSD objectives.
- **Runtime decoding ablation with vLLM:** evaluating a target-response-trained Qwen3 JSD draft while sweeping speculative decoding parameters, especially `gamma`.

**My individual contribution.** I contributed part of both the training-side and runtime-side ablations: on the training side, I compared how data scale and KD objective affect draft-target agreement; on the runtime side, I analyzed how to choose speculative block size so that this agreement is efficiently converted into wall-clock speedup under vLLM.

**Headline result.** RKL/JSD produce the strongest draft-alignment metrics among the Qwen2.5 objectives, but 10k training shows overfitting in loss curves. The 50k setting is slightly better in performance and also more stable, and the Qwen3/vLLM runtime sweep shows that shallow speculation (`gamma=1`) is the best measured operating point.

The notebook is safe to `Run All` by default: it uses embedded precomputed results for all quantitative analysis and does not launch training or inference. It also includes a lightweight Hugging Face asset check that loads a public dataset slice and the relevant models.

The notebook also keeps an optional RCP reproduction mode for rerunning the full pipeline. That mode must be launched under our team's cluster `/scratch` paths. The commands only execute if you manually set both `NOTEBOOK_MODE='rcp_reproduce'` and `EXECUTE_RCP_COMMANDS=True`. Following the project instructions, graders should not expect the notebook to start from `/scratch`, so for grading these two options should remain at their default `False` values.

The first four sections demonstrate the work behind the experiments and provide a guideline for reproducibility.

**If you only want to read the results and analysis, jump directly to Section 5.**

## 1. Environment

Start the RCP job from your laptop with:

```bash
MSYS_NO_PATHCONV=1 MSYS2_ARG_CONV_EXCL="*" bash notebooks/submit.sh milestone
runai workspace port-forward <job-name> --port 8888:8888 -p course-cs-552-sizhang
```

Then open JupyterLab at `http://localhost:8888` with token `cs552`.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'kdsd').exists():
            return p
    raise RuntimeError('Could not find repository root. Open this notebook from inside the repo clone.')


REPO = find_repo_root()
os.chdir(REPO)
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

print('repo:', REPO)
print('This notebook uses embedded precomputed results; it does not run training/inference during grading.')


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Notebook mode controls.
# - "demo": default grading path; run lightweight checks and embedded-result analysis only.
# - "rcp_reproduce": additionally print the full RCP-oriented reproduction commands.
NOTEBOOK_MODE = 'demo'
LOAD_HF_ASSET_CHECK = True
LOAD_QWEN3_TARGET_WEIGHTS = False  # Set True to also load Qwen3-8B weights; config/tokenizer are checked by default.
EXECUTE_RCP_COMMANDS = False  # Keep False for grading; expensive jobs are never launched by default.

# Shared protocols recorded for interpretation.
PROJECT = 'cs552-kdsd'

qwen25_protocol = {
    'target': 'Qwen/Qwen2.5-3B-Instruct',
    'draft_init': 'Qwen/Qwen2.5-0.5B-Instruct',
    'train_data': ['UltraChat 10k', 'UltraChat 50k'],
    'objectives': ['CE', 'FKL', 'RKL', 'JSD'],
    'eval_prompts': 'UltraChat 50k held-out eval split',
    'eval_backend': 'HF/manual loop',
    'gamma': 4,
    'max_new_tokens': 256,
    'temperature': 1.0,
    'top_p': 0.9,
    'n_prompts': 50,
}

qwen3_protocol = {
    'target': 'Qwen/Qwen3-8B',
    'draft': 'epfl-cs552-shallowseek/qwen3-06b-jsd-specdec (best Qwen3-0.6B JSD draft used for the runtime gamma sweep)',
    'eval_backend': 'vLLM',
    'gamma_values': [1, 2, 4, 6, 8],
    'max_new_tokens': 256,
    'temperature': 1.0,
    'top_p': 0.9,
    'n_prompts': 50,
}

print('Project:', PROJECT)
print('NOTEBOOK_MODE:', NOTEBOOK_MODE)
print('All quantitative results below are embedded precomputed metrics.')


## 2. Models, Data, and Baselines

The experiments use public Hugging Face model families and UltraChat-derived splits prepared by the project scripts. For the Qwen3 runtime analysis, the draft model is represented by the public Hugging Face checkpoint `epfl-cs552-shallowseek/qwen3-06b-jsd-specdec`, which is the best draft we selected for the runtime gamma sweep. To keep this deliverable fast and self-contained, the notebook records the model/data identities and analyzes precomputed metrics rather than loading full model weights during grading.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

model_rows = [
    {
        'experiment_stage': 'KD data-scale ablation',
        'target_model': qwen25_protocol['target'],
        'draft_model': qwen25_protocol['draft_init'],
        'data': 'Original UltraChat 10k / 50k',
        'backend': qwen25_protocol['eval_backend'],
    },
    {
        'experiment_stage': 'Runtime gamma sweep',
        'target_model': qwen3_protocol['target'],
        'draft_model': qwen3_protocol['draft'],
        'data': 'Target-generated UltraChat 50k responses',
        'backend': qwen3_protocol['eval_backend'],
    },
]

display(pd.DataFrame(model_rows))
print('Primary baselines: pretrained draft without KD, and vanilla target-only decoding under the same prompt/decode setup.')


### Lightweight Hugging Face Asset Check

To satisfy the requirement that the notebook loads relevant models/datasets correctly, this optional check loads one public UltraChat example, prints an example prompt, and loads the full weights of the selected Qwen3 draft `epfl-cs552-shallowseek/qwen3-06b-jsd-specdec`. For the target `Qwen/Qwen3-8B`, it loads tokenizer/config by default and exposes `LOAD_QWEN3_TARGET_WEIGHTS=True` if a reviewer wants to load the 8B weights as well. The draft is the model used in the runtime gamma sweep and the best draft we selected.

In [ ]:
if LOAD_HF_ASSET_CHECK:
    import torch
    from datasets import load_dataset
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

    hf_dataset = load_dataset('HuggingFaceH4/ultrachat_200k', split='train_sft[:1]')
    example = hf_dataset[0]
    example_prompt = example.get('prompt') or example.get('messages') or str(example)[:500]
    print('Example UltraChat prompt:')
    print(str(example_prompt)[:500])

    qwen3_target_id = 'Qwen/Qwen3-8B'
    qwen3_draft_id = 'epfl-cs552-shallowseek/qwen3-06b-jsd-specdec'
    qwen3_tokenizer = AutoTokenizer.from_pretrained(qwen3_draft_id)
    qwen3_draft_config = AutoConfig.from_pretrained(qwen3_draft_id)
    qwen3_target_config = AutoConfig.from_pretrained(qwen3_target_id)

    model_load_kwargs = {}
    if torch.cuda.is_available():
        model_load_kwargs.update({'dtype': torch.bfloat16, 'device_map': 'auto'})

    qwen3_draft_model = AutoModelForCausalLM.from_pretrained(qwen3_draft_id, **model_load_kwargs).eval()
    qwen3_draft_param_count_b = sum(p.numel() for p in qwen3_draft_model.parameters()) / 1e9
    print(f'Draft model loaded successfully: {qwen3_draft_id} ({qwen3_draft_param_count_b:.2f}B parameters)')

    if LOAD_QWEN3_TARGET_WEIGHTS:
        qwen3_target_model = AutoModelForCausalLM.from_pretrained(qwen3_target_id, **model_load_kwargs).eval()
        qwen3_target_param_count_b = sum(p.numel() for p in qwen3_target_model.parameters()) / 1e9
        print(f'Target model weights loaded successfully: {qwen3_target_id} ({qwen3_target_param_count_b:.2f}B parameters)')
    else:
        qwen3_target_model = None
        qwen3_target_param_count_b = None
        print(f'Target tokenizer/config loaded successfully: {qwen3_target_id}; set LOAD_QWEN3_TARGET_WEIGHTS=True to load full 8B weights.')

    target_loaded = 'config only'
    if qwen3_target_param_count_b is not None:
        target_loaded = f'config/weights, params={qwen3_target_param_count_b:.2f}B'

    hf_asset_rows = [
        {
            'asset': 'UltraChat sample',
            'source': 'HuggingFaceH4/ultrachat_200k',
            'loaded_field_keys': ', '.join(hf_dataset.column_names),
        },
        {
            'asset': 'Qwen3 target tokenizer/config or weights',
            'source': qwen3_target_id,
            'loaded_field_keys': f'hidden_size={qwen3_target_config.hidden_size}, loaded={target_loaded}',
        },
        {
            'asset': 'Qwen3 best runtime draft tokenizer/config/weights',
            'source': qwen3_draft_id,
            'loaded_field_keys': f'vocab={len(qwen3_tokenizer)}, hidden_size={qwen3_draft_config.hidden_size}, params={qwen3_draft_param_count_b:.2f}B',
        },
    ]
    display(pd.DataFrame(hf_asset_rows))
else:
    print('Skipping lightweight Hugging Face asset check because LOAD_HF_ASSET_CHECK=False.')

## 3. Training and Evaluation Parameter Protocols

This section preserves the concrete hyperparameters that were previously embedded in executable command cells. They are shown before the reproduction commands so the reader first sees the protocol and then sees how the scripts implement it.

In [ ]:
parameter_rows = [
    {
        'stage': 'Qwen2.5 KD training',
        'parameter_group': 'data/objective',
        'settings': 'run_loss_size_ablation.sh covers UltraChat 50k non-CE losses={FKL,RKL,JSD}; run_loss_size_ablation_10k.sh covers UltraChat 10k non-CE losses; CE rows were trained separately with scripts/train_size.py using the same budget',
    },
    {
        'stage': 'Qwen2.5 KD training',
        'parameter_group': 'training budget',
        'settings': 'max_steps=8000; save_steps=2000; save_total_limit=4; max_seq_len=512; train_bs=2; grad_accum=4; effective_bs=8',
    },
    {
        'stage': 'Qwen2.5 KD training',
        'parameter_group': 'optimization/loss',
        'settings': 'learning_rate=2e-5; scheduler=cosine; warmup_ratio=0.03; kd_alpha=1.0; kd_temperature=1.0',
    },
    {
        'stage': 'Qwen2.5 HF/manual evaluation',
        'parameter_group': 'decoding/eval',
        'settings': 'backend=manual; eval_prompts=UltraChat 50k held-out split; limit=50; mode=sampling; gamma=4; max_new_tokens=256; temperature=1.0; top_p=0.9; n_warmup=1; n_repeats=3',
    },
    {
        'stage': 'Qwen3 target-generated JSD training',
        'parameter_group': 'data/model/loss',
        'settings': 'target=Qwen/Qwen3-8B; draft_init=Qwen/Qwen3-0.6B; selected_public_draft=epfl-cs552-shallowseek/qwen3-06b-jsd-specdec; data=ultrachat_50k_target_gen; loss=JSD; response_source=target-generated; seed=42',
    },
    {
        'stage': 'Qwen3 vLLM runtime sweep on gamma',
        'parameter_group': 'decoding/eval',
        'settings': 'backend=vllm; gamma={1,2,4,6,8}; temperature=1.0; mode=auto/sampling; top_p=0.9; max_new_tokens=256; n_warmup=1; n_repeats=3; run_vanilla_baseline=true; prompt_limit=50',
    },
]

display(pd.DataFrame(parameter_rows))

## 4. Reproducibility Entry Points

The expensive jobs were completed before notebook submission. In default `demo` mode, this section only lists repository-relative scripts used to produce the cached metrics. If `NOTEBOOK_MODE='rcp_reproduce'`, it restores the RCP-oriented `/scratch` paths and prints commands that can reproduce the training/evaluation flow. If `EXECUTE_RCP_COMMANDS=True`, those commands are executed with bash, so this option should only be used intentionally inside an RCP pod.

In [ ]:
repro_rows = [
    {
        'stage': 'Qwen2.5 KD data-scale ablation',
        'repo_entry_point': 'scripts/run_loss_size_ablation.sh',
        'purpose': 'Train/evaluate non-CE KD objectives on UltraChat 50k.',
    },
    {
        'stage': 'Qwen2.5 KD data-scale ablation',
        'repo_entry_point': 'scripts/run_loss_size_ablation_10k.sh',
        'purpose': 'Train/evaluate non-CE KD objectives on UltraChat 10k.',
    },
    {
        'stage': 'Qwen2.5 CE baselines',
        'repo_entry_point': 'scripts/train_size.py',
        'purpose': 'Train CE 10k/50k baselines separately with the same training budget as the KD runs.',
    },
    {
        'stage': 'Qwen3 target-generated JSD draft',
        'repo_entry_point': 'scripts/run_qwen3_loss_sweep.sh',
        'purpose': 'Train the Qwen3 JSD draft used by the runtime sweep.',
    },
    {
        'stage': 'Qwen3 vLLM runtime sweep on gamma',
        'repo_entry_point': 'scripts/submit_qwen3_runtime_sweep_interactive.sh',
        'purpose': 'Launch the gamma-only vLLM runtime evaluation.',
    },
]

display(pd.DataFrame(repro_rows))

qwen3_checkpoint_run = 'qwen3_8btarget_0p6b_tgen_jsd_ultrachat_50k_target_gen_seed42'
qwen3_draft_path = str(REPO / 'checkpoints' / qwen3_checkpoint_run / 'model')
qwen3_experiment = f'qwen3_runtime_sweep_{qwen3_checkpoint_run}'

rcp_command_templates = [
    'export KDSD_VENV=/scratch/venvs/kdsd-vllm && source scripts/env.sh && echo "$KDSD_PYTHON"',
    'source scripts/env.sh && "$KDSD_PYTHON" -m pip install -e . --quiet',
    'source scripts/env.sh && "$KDSD_PYTHON" scripts/prepare_data.py data=ultrachat_50k',
    'WANDB=false EVAL_BACKEND=manual bash scripts/run_loss_size_ablation.sh',
    'WANDB=false EVAL_BACKEND=manual bash scripts/run_loss_size_ablation_10k.sh',
    'source scripts/env.sh && DATA=ultrachat_50k_target_gen LOSSES=jsd RUN_NAME_PREFIX=qwen3_8btarget_0p6b_tgen TARGET_ID=Qwen/Qwen3-8B RUN_EVAL=false EVAL_BACKEND=vllm EVAL_REPORT_TO_WANDB=false bash scripts/run_qwen3_loss_sweep.sh',
    (
        'WANDB_MODE=offline EVAL_REPORT_TO_WANDB=false '
        f'CHECKPOINT_RUN={qwen3_checkpoint_run} '
        f'DRAFT_PATH={qwen3_draft_path} '
        'DRAFT_SIZE=0.6b DATA=ultrachat_50k_target_gen BASE_DATA=ultrachat_50k '
        'TARGET_ID=Qwen/Qwen3-8B GAMMAS="1 2 4 6 8" TEMPERATURES="1.0" '
        'EVAL_MAX_NEW_TOKENS_VALUES="256" EVAL_WARMUP=1 EVAL_REPEATS=3 '
        'EVAL_BACKEND=vllm EVAL_MODE=auto EVAL_TOP_P=0.9 '
        'EVAL_RUN_VANILLA_BASELINE=true '
        'EVAL_PROMPTS_JSONL=/scratch/cs552-data/processed/ultrachat_50k/eval.jsonl '
        'EVAL_PROMPTS_LIMIT=50 RESULTS_ROOT=/scratch/cs552-results-syz '
        f'EXPERIMENT_NAME={qwen3_experiment}_gamma_only '
        f'SUMMARY_CSV=/scratch/cs552-results-syz/{qwen3_experiment}_gamma_only.csv '
        'bash scripts/submit_qwen3_runtime_sweep_interactive.sh'
    ),
]

def run_rcp_command(command: str) -> None:
    print('\n$ ' + command)
    subprocess.run(command, shell=True, check=True, executable='/bin/bash')

if NOTEBOOK_MODE == 'rcp_reproduce':
    print('RCP reproduction commands:')
    for command in rcp_command_templates:
        print('$', command)
    if EXECUTE_RCP_COMMANDS:
        for command in rcp_command_templates:
            run_rcp_command(command)
else:
    print("Default demo mode: RCP commands are not printed or executed. Set NOTEBOOK_MODE='rcp_reproduce' to display them.")


## 5. Results: Data Scale, Objective Choice, and Draft Quality

This section is the main results entry point. The 10k/50k ablation is summarized from embedded completed metrics rather than runtime file reads. Every row uses the same HF/manual-loop protocol: `ultrachat_50k/eval.jsonl`, 50 prompts, `gamma=4`, `max_new_tokens=256`, sampling with `temperature=1.0`, `top_p=0.9`, `n_warmup=1`, and `n_repeats=3`.

Headline takeaways:

- RKL and JSD improve acceptance-oriented metrics more consistently than CE/FKL.
- Scaling from 10k to 50k gives modest metric gains, but it matters more for training stability.
- HF/manual-loop speedup remains below 1x, so this table should be read as a draft-quality attribution rather than the final serving-speed result.

| model | train data | objective | acceptance rate | avg accepted tokens | speedup | tokens/s |
|---|---:|---|---:|---:|---:|---:|
| pretrained Qwen2.5-0.5B | none | pretrained | 0.450 | 1.801 | 0.508 | 18.080 |
| CE 10k | UltraChat 10k | CE | 0.423 | 1.694 | 0.486 | 17.748 |
| FKL 10k | UltraChat 10k | FKL | 0.447 | 1.788 | 0.509 | 17.865 |
| RKL 10k | UltraChat 10k | RKL | 0.463 | 1.851 | 0.508 | 18.506 |
| JSD 10k | UltraChat 10k | JSD | 0.469 | 1.876 | 0.524 | 18.849 |
| CE 50k | UltraChat 50k | CE | 0.429 | 1.714 | 0.496 | 17.826 |
| FKL 50k | UltraChat 50k | FKL | 0.451 | 1.804 | 0.509 | 17.687 |
| RKL 50k | UltraChat 50k | RKL | 0.472 | 1.888 | 0.521 | 18.773 |
| JSD 50k | UltraChat 50k | JSD | 0.471 | 1.886 | 0.524 | 18.045 |

The short version: KD helps more than simply scaling from 10k to 50k. CE underperforms the pretrained draft, FKL is close to the baseline, and RKL/JSD are the strongest objectives.

In [ ]:
# Self-contained 10k vs 50k analysis from the completed results above.
# This cell does not read JSON files and can run without executing previous cells.
import matplotlib.pyplot as plt
import pandas as pd

rows = [
    {
        'model': 'pretrained Qwen2.5-0.5B',
        'train_data': 'pretrained',
        'objective': 'pretrained',
        'acceptance_rate': 0.4503466557911909,
        'avg_accepted_tokens': 1.8013866231647635,
        'speedup': 0.5075373747452575,
        'tokens_per_second': 18.079710466387365,
        'sd_time_s': 628.9370629657058,
        'vanilla_time_s': 319.20906581760704,
    },
    {
        'model': 'CE 10k',
        'train_data': '10k',
        'objective': 'CE',
        'acceptance_rate': 0.4234769687964339,
        'avg_accepted_tokens': 1.6939078751857355,
        'speedup': 0.4857984885301736,
        'tokens_per_second': 17.74846967864597,
        'sd_time_s': 643.2103841456897,
        'vanilla_time_s': 312.4706324248884,
    },
    {
        'model': 'FKL 10k',
        'train_data': '10k',
        'objective': 'FKL',
        'acceptance_rate': 0.4469641251940835,
        'avg_accepted_tokens': 1.787856500776334,
        'speedup': 0.5087557905017256,
        'tokens_per_second': 17.864969865772164,
        'sd_time_s': 632.6906837752709,
        'vanilla_time_s': 321.8850489671653,
    },
    {
        'model': 'RKL 10k',
        'train_data': '10k',
        'objective': 'RKL',
        'acceptance_rate': 0.4628702716008862,
        'avg_accepted_tokens': 1.8514810864035447,
        'speedup': 0.5076301159654848,
        'tokens_per_second': 18.505908177155526,
        'sd_time_s': 622.2337152961021,
        'vanilla_time_s': 315.8645730533948,
    },
    {
        'model': 'JSD 10k',
        'train_data': '10k',
        'objective': 'JSD',
        'acceptance_rate': 0.4691166610794502,
        'avg_accepted_tokens': 1.8764666443178009,
        'speedup': 0.5239097536536697,
        'tokens_per_second': 18.84895456464947,
        'sd_time_s': 603.3225853983314,
        'vanilla_time_s': 316.08658708973485,
    },
    {
        'model': 'CE 50k',
        'train_data': '50k',
        'objective': 'CE',
        'acceptance_rate': 0.42853448961731244,
        'avg_accepted_tokens': 1.7141379584692498,
        'speedup': 0.495899984514268,
        'tokens_per_second': 17.825759308531637,
        'sd_time_s': 634.0262877099836,
        'vanilla_time_s': 314.4136262570197,
    },
    {
        'model': 'FKL 50k',
        'train_data': '50k',
        'objective': 'FKL',
        'acceptance_rate': 0.4510233319688907,
        'avg_accepted_tokens': 1.8040933278755629,
        'speedup': 0.5094669917174816,
        'tokens_per_second': 17.68676113062409,
        'sd_time_s': 642.1752358227967,
        'vanilla_time_s': 327.16708555010456,
    },
    {
        'model': 'RKL 50k',
        'train_data': '50k',
        'objective': 'RKL',
        'acceptance_rate': 0.47206447302488724,
        'avg_accepted_tokens': 1.888257892099549,
        'speedup': 0.5211662271273737,
        'tokens_per_second': 18.772881527663206,
        'sd_time_s': 610.5615689903498,
        'vanilla_time_s': 318.2040693396703,
    },
    {
        'model': 'JSD 50k',
        'train_data': '50k',
        'objective': 'JSD',
        'acceptance_rate': 0.47146819060425244,
        'avg_accepted_tokens': 1.8858727624170097,
        'speedup': 0.5237972144475006,
        'tokens_per_second': 18.04471667565907,
        'sd_time_s': 630.7109279998888,
        'vanilla_time_s': 330.36462720793986,
    },
]

objective_order = ['CE', 'FKL', 'RKL', 'JSD']
df = pd.DataFrame(rows)
df['objective'] = pd.Categorical(df['objective'], ['pretrained', *objective_order], ordered=True)
df = df.sort_values(['objective', 'train_data']).reset_index(drop=True)

pretrained = df[df['train_data'] == 'pretrained'].iloc[0]
paired = df[df['train_data'].isin(['10k', '50k'])].copy()
paired['acceptance_delta_vs_pretrained'] = paired['acceptance_rate'] - pretrained['acceptance_rate']
paired['speedup_delta_vs_pretrained'] = paired['speedup'] - pretrained['speedup']

comparison = paired.pivot(index='objective', columns='train_data', values='acceptance_rate').loc[objective_order]
comparison['50k_minus_10k_acceptance'] = comparison['50k'] - comparison['10k']

print('Unified eval protocol: ultrachat_50k/eval.jsonl, 50 prompts, gamma=4, max_new_tokens=256, n_repeats=3')
print('Pretrained acceptance baseline:', round(pretrained['acceptance_rate'], 4))
display(
    paired[
        [
            'objective',
            'train_data',
            'acceptance_rate',
            'avg_accepted_tokens',
            'speedup',
            'tokens_per_second',
            'acceptance_delta_vs_pretrained',
        ]
    ].sort_values(['objective', 'train_data'])
)

display(comparison)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
metrics = [
    ('acceptance_rate', 'Acceptance Rate'),
    ('avg_accepted_tokens', 'Avg Accepted Tokens'),
    ('tokens_per_second', 'Tokens / Second'),
]

for ax, (metric, title) in zip(axes, metrics):
    pivot = paired.pivot(index='objective', columns='train_data', values=metric).loc[objective_order]
    pivot.plot(kind='bar', ax=ax, width=0.75)
    ax.axhline(pretrained[metric], color='black', linestyle='--', linewidth=1, label='pretrained')
    ax.set_title(title)
    ax.set_xlabel('Objective')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()

plt.tight_layout()
plt.show()

best_acceptance = paired.loc[paired['acceptance_rate'].idxmax()]
best_speedup = paired.loc[paired['speedup'].idxmax()]
print(
    f"Best acceptance: {best_acceptance['objective']} {best_acceptance['train_data']} "
    f"({best_acceptance['acceptance_rate']:.3f})."
)
print(
    f"Best HF-loop speedup: {best_speedup['objective']} {best_speedup['train_data']} "
    f"({best_speedup['speedup']:.3f}x)."
)
print(
    'Main pattern: RKL/JSD improve acceptance over CE and pretrained; 50k is slightly better for RKL, '
    'while JSD is already strong at 10k under this eval protocol.'
)

## 6. 10k vs 50k Comparison

Increasing from UltraChat 10k to 50k improves acceptance for every objective, but only modestly: CE `+0.005`, FKL `+0.004`, RKL `+0.009`, and JSD `+0.002`. RKL benefits most from the larger dataset, while JSD is already strong at 10k.

The objective choice is more important than the data-size change in this slice. CE-only training is consistently worse than the pretrained draft, which suggests that simply fine-tuning on labels does not necessarily improve draft-target alignment. In speculative decoding, the draft is useful only when its proposed next-token distribution overlaps with the target distribution. FKL, RKL, and JSD optimize that overlap more directly than CE, and RKL/JSD most clearly improve acceptance and average accepted tokens.

The main conclusion is that the KD objective dominates the data-scale effect in this setup. RKL 50k gives the best acceptance (`0.472`), and JSD gives the best HF-loop speedup (`~0.524x`). The absolute HF-loop speedups remain below 1x, so I treat this first part mainly as a draft-quality ablation rather than a final serving-speed measurement.

### Overfitting Check: RKL 10k vs 50k

We observed overfitting trends across multiple objectives/models when training on the 10k dataset; RKL is shown here as a representative case. These RKL curves explain why the 50k setting is a safer draft-training regime. In the 10k run, the training loss keeps improving while the evaluation loss stops improving and begins to drift upward. That divergence means the draft is becoming better at the finite training prompts but worse at matching held-out target behavior, which is exactly the failure mode speculative decoding cannot use. In contrast, the 50k run has a much more stable eval-loss trajectory relative to train loss, suggesting that the larger data scale reduces overfitting and gives the KD objective more coverage of target behavior.

<table>
<tr><th colspan="2">RKL 10k Training Run</th></tr>
<tr>
<td><img src="../scripts/overfitting_analysis/rkl_10K_train_loss.png" alt="RKL 10k train loss" width="420"/></td>
<td><img src="../scripts/overfitting_analysis/rkl_10K_eval_loss.png" alt="RKL 10k eval loss" width="420"/></td>
</tr>
<tr><th colspan="2">RKL 50k Training Run</th></tr>
<tr>
<td><img src="../scripts/overfitting_analysis/rkl_50K_train_loss.png" alt="RKL 50k train loss" width="420"/></td>
<td><img src="../scripts/overfitting_analysis/rkl_50K_eval_loss.png" alt="RKL 50k eval loss" width="420"/></td>
</tr>
</table>

This supports a more nuanced reading of the metric table: 10k can already produce a strong RKL/JSD draft under the fixed HF evaluation, but the loss curves show less reliable generalization. The practical selection rule is therefore two-stage: first choose an objective that improves target agreement, then check whether the training curve generalizes instead of just memorizing the smaller split. The 50k setting is preferable when selecting a robust draft checkpoint because it preserves the acceptance gains while avoiding the clearer overfitting pattern seen at 10k.

## 7. Transition to Runtime Tuning

The 10k/50k ablation above is one controlled slice of the broader project. Combined with the rest of the team's evaluations, the strongest draft setting was target-generated response training with JSD. I therefore use that setting as the starting point for the runtime part instead of repeating the full loss/data sweep here.

The runtime stage also changes the evaluation stack. As discussed in the final report, Qwen2.5-3B/0.5B is useful for controlled KD ablations, but it leaves limited room for final wall-clock SD gains: the target is relatively small and manual-loop overhead is visible. For the runtime sweep, I switch to vLLM and a larger target-draft gap: Qwen3-8B with our selected Qwen3-0.6B JSD draft, published as `epfl-cs552-shallowseek/qwen3-06b-jsd-specdec`.

The next section asks a systems question: given a strong target-generated JSD draft, which vLLM `gamma` setting actually converts acceptance into wall-clock speedup?

## 8. vLLM Runtime Ablation on a Target-Generated JSD Draft

The second part of my contribution studies how the runtime choice of `gamma` affects a stronger target-generated draft. Here the model family changes from the earlier Qwen2.5 HF-loop experiments to a Qwen3 vLLM setup:

- target: `Qwen/Qwen3-8B`
- draft: `epfl-cs552-shallowseek/qwen3-06b-jsd-specdec` (selected best Qwen3-0.6B JSD draft for the runtime gamma sweep)
- training data: target-generated UltraChat 50k responses
- objective: JSD
- backend: vLLM draft-model speculative decoding

The full project sweep contains multiple `gamma`, `temperature`, and `max_new_tokens` settings. My main responsibility here is the `gamma` scan. Therefore, for a focused notebook analysis, I fix:

- `max_new_tokens = 256`
- `runtime_temperature = 1.0`

and compare `gamma in {1, 2, 4, 6, 8}`. This isolates the core speculative-decoding question: how many draft tokens should be proposed per verification step?

Mechanistically, `gamma` trades target-call savings against wasted draft work. A larger `gamma` can reduce the number of target verifications only if the later drafted positions are accepted often enough. Once rejection happens early in the block, the remaining drafted tokens have already been computed but cannot be used, so high `gamma` can reduce throughput even while average accepted tokens per outer step increases.

In [ ]:
# Qwen3 vLLM runtime sweep analysis.
# Self-contained grading version: the five rows used below are embedded directly
# from the completed sweep, so this notebook does not depend on external CSV files.

import pandas as pd
import matplotlib.pyplot as plt

focus = pd.DataFrame([
    {
        'runtime_mode': 'sampling',
        'runtime_temperature': 1.0,
        'gamma': 1,
        'max_new_tokens': 256,
        'speedup': 1.1560925347714506,
        'acceptance_rate': 0.7072014893797072,
        'avg_accepted_tokens': 0.7072014893797072,
        'tokens_per_second': 597.8053921403215,
        'sd_time_s': 21.411650293370865,
        'vanilla_time_s': 24.753849061302997,
        'vllm_num_drafts': 23634,
        'vllm_num_draft_tokens': 23634,
        'vllm_num_accepted_tokens': 16714,
    },
    {
        'runtime_mode': 'sampling',
        'runtime_temperature': 1.0,
        'gamma': 2,
        'max_new_tokens': 256,
        'speedup': 1.0068717480699172,
        'acceptance_rate': 0.5977233210171702,
        'avg_accepted_tokens': 1.1954466420343404,
        'tokens_per_second': 521.1042270631478,
        'sd_time_s': 24.563224274994962,
        'vanilla_time_s': 24.732016563997604,
        'vllm_num_drafts': 18404,
        'vllm_num_draft_tokens': 36808,
        'vllm_num_accepted_tokens': 22001,
    },
    {
        'runtime_mode': 'sampling',
        'runtime_temperature': 1.0,
        'gamma': 4,
        'max_new_tokens': 256,
        'speedup': 0.5889439991852603,
        'acceptance_rate': 0.4210822998872604,
        'avg_accepted_tokens': 1.6843291995490417,
        'tokens_per_second': 304.5398394566998,
        'sd_time_s': 42.03062569033742,
        'vanilla_time_s': 24.753684782326065,
        'vllm_num_drafts': 15079,
        'vllm_num_draft_tokens': 60316,
        'vllm_num_accepted_tokens': 25398,
    },
    {
        'runtime_mode': 'sampling',
        'runtime_temperature': 1.0,
        'gamma': 6,
        'max_new_tokens': 256,
        'speedup': 0.4346949671408115,
        'acceptance_rate': 0.33027916789323913,
        'avg_accepted_tokens': 1.9816750073594347,
        'tokens_per_second': 224.9195078562197,
        'sd_time_s': 56.90924776601605,
        'vanilla_time_s': 24.738163587656647,
        'vllm_num_drafts': 13588,
        'vllm_num_draft_tokens': 81528,
        'vllm_num_accepted_tokens': 26927,
    },
    {
        'runtime_mode': 'sampling',
        'runtime_temperature': 1.0,
        'gamma': 8,
        'max_new_tokens': 256,
        'speedup': 0.3420226534619635,
        'acceptance_rate': 0.2580020406620815,
        'avg_accepted_tokens': 2.064016325296652,
        'tokens_per_second': 176.87392517134936,
        'sd_time_s': 72.36793092933173,
        'vanilla_time_s': 24.751471762002137,
        'vllm_num_drafts': 13231,
        'vllm_num_draft_tokens': 105848,
        'vllm_num_accepted_tokens': 27309,
    },
]).sort_values('gamma').reset_index(drop=True)

# Derived systems metrics make the gamma trade-off easier to see.
focus['draft_tokens_per_accepted_token'] = focus['vllm_num_draft_tokens'] / focus['vllm_num_accepted_tokens']
focus['sd_time_delta_vs_gamma1_s'] = focus['sd_time_s'] - focus.loc[focus['gamma'] == 1, 'sd_time_s'].iloc[0]
focus['speedup_delta_vs_gamma1'] = focus['speedup'] - focus.loc[focus['gamma'] == 1, 'speedup'].iloc[0]

cols = [
    'runtime_mode',
    'runtime_temperature',
    'gamma',
    'max_new_tokens',
    'speedup',
    'acceptance_rate',
    'avg_accepted_tokens',
    'tokens_per_second',
    'sd_time_s',
    'vanilla_time_s',
    'vllm_num_drafts',
    'vllm_num_draft_tokens',
    'vllm_num_accepted_tokens',
]

summary_cols = [
    'gamma',
    'speedup',
    'tokens_per_second',
    'acceptance_rate',
    'avg_accepted_tokens',
    'vllm_num_draft_tokens',
    'vllm_num_accepted_tokens',
    'draft_tokens_per_accepted_token',
    'sd_time_s',
    'sd_time_delta_vs_gamma1_s',
]

runtime_table = focus[summary_cols].copy()
for col in ['speedup', 'tokens_per_second', 'acceptance_rate', 'avg_accepted_tokens', 'draft_tokens_per_accepted_token', 'sd_time_s', 'sd_time_delta_vs_gamma1_s']:
    runtime_table[col] = runtime_table[col].round(3)

display(runtime_table)

best_speedup = focus.loc[focus['speedup'].idxmax()]
best_acceptance = focus.loc[focus['acceptance_rate'].idxmax()]
print(
    f"Best speedup in this slice: gamma={int(best_speedup['gamma'])}, "
    f"speedup={best_speedup['speedup']:.3f}x, "
    f"acceptance={best_speedup['acceptance_rate']:.3f}."
)
print(
    f"Best acceptance in this slice: gamma={int(best_acceptance['gamma'])}, "
    f"acceptance={best_acceptance['acceptance_rate']:.3f}, "
    f"speedup={best_acceptance['speedup']:.3f}x."
)

In [ ]:
# Visualize the gamma trade-off for the fixed max_new=256, temperature=1.0 slice.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

plot_specs = [
    ('speedup', 'vLLM Speedup (vanilla / speculative)', 'Speedup'),
    ('acceptance_rate', 'Draft Acceptance Rate', 'Acceptance'),
    ('avg_accepted_tokens', 'Average Accepted Tokens per Draft Step', 'Avg accepted'),
    ('tokens_per_second', 'Speculative Throughput', 'Tokens/s'),
]

for ax, (metric, title, ylabel) in zip(axes, plot_specs):
    ax.plot(focus['gamma'], focus[metric], marker='o', linewidth=2)
    best_idx = focus[metric].idxmax()
    ax.scatter([focus.loc[best_idx, 'gamma']], [focus.loc[best_idx, metric]], s=80, zorder=3)
    ax.set_title(title)
    ax.set_xlabel('gamma')
    ax.set_ylabel(ylabel)
    ax.set_xticks(focus['gamma'])
    ax.grid(alpha=0.3)

plt.suptitle('Qwen3 target-generated JSD draft: gamma sweep at temperature=1.0, max_new_tokens=256')
plt.tight_layout()
plt.show()

# Attribute the slowdown to wasted draft work and extra SD time.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(focus['gamma'], focus['draft_tokens_per_accepted_token'], marker='o', linewidth=2, color='#C44E52')
ax.set_title('Draft Work per Accepted Token')
ax.set_xlabel('gamma')
ax.set_ylabel('draft tokens / accepted token')
ax.set_xticks(focus['gamma'])
ax.grid(alpha=0.3)
for _, row in focus.iterrows():
    ax.annotate(f"{row['draft_tokens_per_accepted_token']:.2f}", (row['gamma'], row['draft_tokens_per_accepted_token']), textcoords='offset points', xytext=(0, 7), ha='center', fontsize=8)

ax = axes[1]
sc = ax.scatter(
    focus['draft_tokens_per_accepted_token'],
    focus['speedup'],
    c=focus['sd_time_delta_vs_gamma1_s'],
    s=95,
    cmap='viridis_r',
    edgecolor='white',
)
for _, row in focus.iterrows():
    ax.annotate(f"g={int(row['gamma'])}", (row['draft_tokens_per_accepted_token'], row['speedup']), textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.axhline(1.0, color='black', linewidth=0.9, linestyle='--', label='break-even')
ax.set_title('Speedup Falls as Draft Waste Grows')
ax.set_xlabel('draft tokens / accepted token')
ax.set_ylabel('speedup')
ax.grid(alpha=0.3)
ax.legend()
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('extra SD time vs gamma=1 (s)')

plt.suptitle('Runtime Cost Attribution: accepted-token yield vs draft overhead', y=1.03)
plt.tight_layout()
plt.show()

### Runtime Interpretation

This vLLM sweep shows why runtime tuning is a real systems variable, not a cosmetic decode flag. A larger `gamma` lets the draft propose more tokens before each target verification, so in principle it can reduce target calls. That benefit only materializes when the later draft positions are accepted often enough to offset the extra draft forward work and verification overhead.

The first visualization shows the direct runtime trade-off. At `gamma=1`, the run is fastest: `1.156x` speedup and about `598 tokens/s`. Moving to `gamma=2` keeps speedup near break-even (`1.007x`) and increases average accepted tokens from `0.707` to `1.195`, but throughput already drops. By `gamma=4`, `6`, and `8`, average accepted tokens continue to increase, yet acceptance rate falls from `0.707` to `0.421`, `0.330`, and `0.258`, while speedup drops to `0.589x`, `0.435x`, and `0.342x`. This shows why average accepted tokens alone is misleading: a longer speculative block can accept more tokens per outer step without improving end-to-end runtime.

The second visualization explains why the slowdown happens. A useful attribution view is `speedup ≈ token_yield / runtime_cost`: KD improves the numerator by making the target accept more draft tokens, while `gamma` changes both the numerator and the cost. As `gamma` grows, the number of drafted tokens needed for each accepted token increases sharply; points with more draft waste move to lower speedup and higher extra SD time relative to `gamma=1`. At large `gamma`, draft computation and rejection handling grow faster than the useful accepted-token yield, so throughput drops.

It also separates training from inference. The JSD draft was trained once on target-generated responses; `gamma` is not in the training loss. The gamma sweep is therefore an inference-time policy search over how to spend compute. For this target/draft pair and vLLM backend, the best policy in the focused slice is `gamma=1`, with `gamma=2` as a borderline setting and larger values clearly dominated. This is why runtime tuning is part of the final system design, not an afterthought after KD training.

## 9. How the Two Contributions Fit Together

The two analyses answer different questions:

- The **Qwen2.5 HF/manual-loop data-scale ablation** asks whether more original UltraChat training data improves draft-target agreement under a fixed speculative decoding protocol. This is a training-data question.
- The **Qwen3 vLLM runtime sweep on gamma** asks how the same trained draft behaves under different decoding-time `gamma` settings. This is an inference-systems question.

The important lesson is that these axes are complementary. Training objective and data scale determine the draft distribution, while `gamma`, temperature, and backend implementation determine how much of that distribution can be converted into wall-clock speedup. A model can have better acceptance but still fail to speed up if the runtime setting makes draft verification too expensive.

Takeaway for the project: practical speculative decoding needs both a well-aligned draft and a low-overhead serving policy. My results support using distribution-matching KD objectives such as RKL/JSD, preferring the more stable 50k setting over the visibly overfit 10k runs, and tuning `gamma` under the actual backend rather than assuming that larger speculative blocks are always better.